# Step 6 : forward pass と Line Search

Step 5 の backward pass で求めた入力修正則に基づいて、非線形動力学モデルをrolloutし、新しい状態軌道を求める。その状態軌道を候補軌道として、Line Search により基準軌道で求まるコスト$\bar{J}$よりも候補軌道から求めるコスト$J^{\text{cand}}$が小さいを候補軌道を探索し、$\bar{J} > J^{\text{cand}}$となった候補軌道を新しい基準軌道とする。

## 1. forward pass の考え方

#### 基準軌道と候補軌道

前回の反復で得た状態軌道が以下のように与えられているとする。右上の$(i)$は$i$回目の反復であることを示す。<br>
ここで基準軌道とは、最小コストを得られている軌道という意味ではなく、現在の反復において局所近似の基準として用いている軌道を意味する。

$$
(\bar{X}^{(i)}, \bar{U}^{(i)})
$$

基準軌道は時系列であるため、次のように配列で表現する。

$$
\bar{X}^{(i)} = \left[\bar{X}_0, \bar{X}_1, \cdots , \bar{X}_N \right]
$$

$$
\bar{U}^{(i)} = \left[\bar{u}_0, \bar{u}_1, \cdots , \bar{u}_{N-1} \right]
$$

これに対し、forward pass によって新しく生成する候補軌道を右上に"new"をつけて以下で表す。

$$
(X^{\text{new}}, U^{\text{new}})
$$

$$
{X}^{\text{new}} = \left[{X}_0^{\text{new}}, {X}_1^{\text{new}}, \cdots , {X}_N^{\text{new}} \right]
$$

$$
{U}^{\text{new}} = \left[{u}_0^{\text{new}}, {u}_1^{\text{new}}, \cdots , {u}_{N-1}^{\text{new}} \right]
$$

よって、以下の意味となる。

- $\bar{X}_n$ : 現在の基準軌道における時刻$n$の状態
- $\bar{u}_n$ : 現在の基準軌道における時刻$n$の入力
- $X_n^{\text{new}}$ : forward pass で生成中の候補軌道における時刻$n$の状態
- $u_n^{\text{new}}$ : forward pass で生成中の候補軌道における時刻$n$の入力

#### 入力摂動と状態摂動

基準入力と候補入力の差を、入力摂動とする。

$$
\delta u_n = u_n^{\text{new}} - \bar{u}_n
$$

基準状態と候補状態の差を、状態摂動とする。

$$
\delta X_n = X_n^{\text{new}} - \bar{X}_n
$$

#### iLQRの反復について

ここで、iLQRでなぜ反復が必要かを再度説明する。

扱いたい問題は、非線形なQ関数$Q_n$の最小化問題である。

$$
Q_n (X_n , u_n) = \ell_n(X_n, u_n) + V_{n+1}(f(X_n, u_n))
$$

しかし、この非線形Q関数を直接最小化するのは難しいため、基準点近傍でQ関数の二次近似を行い、局所二次関数$\tilde{Q}_n$を求めた。

$$
Q_n (\bar{X}_n + \delta X_n, \bar{u}_n + \delta u_n) \approx \tilde{Q}_n (\delta X_n , \delta u_n)
$$

そして、Step 5 で求めた入力摂動$\delta u_n^{\star}$は基準点 ($\bar{X}_n, \bar{u}_n$) 近傍の$\tilde{Q}_n$を最小化しており、非線形なQ関数$Q_n$を厳密に最小化は行えていない。

$$
\delta u_n^{\star} = k_n + K_n \delta X_n
$$

Q関数は、現在のステージコストと将来の最小コストを扱う問題であるため、入力摂動$\delta u_n^{\star}$により、時刻$n$を含むコストを最小化する、つまり、基準点($\bar{X}_n, \bar{u}_n$)の近傍点でよりコストが小さい候補点(${X}_n^{\text{new}}, {u}_n^{\text{new}}$)を探している。

よって、$\delta u_n = u_n^{\text{new}} - \bar{u}_n$の関係から、現在の状態摂動 $\delta X_n$ を用いて次のような候補入力が計算される。

$$
u_n^{\text{new}} = \bar{u}_n + \delta u_n^{\star} = \bar{u}_n + k_n + K_n \delta X_n
$$

この$u_n^{\text{new}}$は時刻$n$時点でコストが基準点($\bar{X}_n, \bar{u}_n$)よりも小さいと予想される候補点(${X}_n^{\text{new}}, {u}_n^{\text{new}}$)の$u_n^{\text{new}}$である。
これは、$u_n^{\text{new}}$は与えられた状態摂動に$\delta X_n$に対して局所二次近似Q関数$\tilde{Q}_n$を最小化するように計算されたものであり、元の非線形Q関数 $Q_n$ や軌道全体の総コストが、実際に小さくなることはまだ保証されていないことを意味している。

時刻$n$時点の候補状態${X}_n^{\text{new}}$は同じ時刻の候補入力$u_n^{\text{new}}$によって決まるのではなく、一時刻前の$n-1$の候補状態と候補入力を次のように非線形動力学モデルへ適用した結果として計算されている。

$$
{X}_n^{\text{new}} = f({X}_{n-1}^{\text{new}}, u_{n-1}^{\text{new}})
$$

$k_n$ は目標状態と基準状態の差が考慮された値であり、状態摂動がなくとも $\bar{u}_n$ を変動させる。よって、例えば $\bar{u}_n=0$ でも $k_n$ による入力修正が行われるため、フィードフォワードのように入力を与える。また、状態摂動による $K_n \delta X_n$ による入力修正はフィードバックのようになる。

これらの入力修正を行っても、あくまで最小化しているのは、局所二次近似を行った $\tilde{Q}_n$ であり、元の非線形Q関数ではない。場合によっては、$k_n$ により、元の非線形Q関数のコストが増えることも考えられる。

そこで、一回の反復では、現在の基準軌道周辺で構成した局所二次問題に基づいて、よりコストが小さくなると予想される候補軌道(${X}^{\text{new}}, {U}^{\text{new}}$)を生成する。そして、この候補軌道で総コストを計算する。

この反復では現在の基準軌道 ($\bar{X}^{(i)}, \bar{u}^{(i)}$) 周辺で求めた以下のパラメータを用いて入力修正則の係数 $k_n, K_n$ を構成している。

$$
\phi_X, \phi_XX, \ell_{X}, \ell_{u},\ell_{XX}, \ell_{uX},\ell_{uu}, A, B
$$

よって、これらを用いてコストがより小さい候補軌道 (${X}^{\text{new}}, {U}^{\text{new}}$) を構成できたとして、この候補軌道で求めた上記のパラメータにより入力修正則を再度求めると、上記の値とは異なる。そこで、コストが収束するまで反復を繰り返し、局所最適な軌道を求める。「局所」最適とは、一般にすべての軌道パターンの中から最も最適な軌道を求めているのではなく、元の軌道付近で最適な軌道となっているという意味である。

## 2. Line Search の考え方

Step5で求めた入力修正則をそのまま適用する場合、新しい候補入力は次のようになる。

$$
u_n^{\text{new}} = \bar{u}_n + \delta u_n^{\star} = \bar{u}_n + k_n + K_n \delta X_n
$$

ただし、この入力修正則が最小化するのは、現在の基準軌道周辺で構成した局所二次Q関数$\tilde{Q}_n$であり、元の非線形Q関数や軌道全体の総コストが、実際に小さくなることは保証されていない。

そこで、入力修正則の係数$k_n, K_n$を調整すること考える。

まず、$k_n$は$\tilde{Q}_n$を最小化する方向へ入力摂動を更新する入力修正量である。<br>
そして、$K_n$はその更新によって候補状態が基準状態とずれたときに生じる状態摂動$\delta X_n$に応じ、局所二次Q関数$\tilde{Q}_n$を最小化するように入力摂動を調整する入力修正量である。

#### $k_n$ による$\tilde{Q}_n$の最小化について

基準点近傍の局所二次近似$\tilde{Q}_n$は以下の式である。

$$
\begin{aligned}
\tilde{Q}_n(\delta X_n, \delta u_n) &= \bar{Q}_n + Q_{X,n}^T \delta X_n + Q_{u,n}^T \delta u_n \\
& + \frac{1}{2} \delta X_n^T Q_{XX,n} \delta X_n + \delta u_n^T Q_{uX,n} \delta X_n + \frac{1}{2} \delta u_n^T Q_{uu,n} \delta u_n
\end{aligned}
$$

上式に $\delta u_n = k_n$ のみを適用すると、次のようになる。

$$
\begin{aligned}
\tilde{Q}_n (\delta X_n = 0, k_n)&= \bar{Q}_n + Q_{u,n}^T k_n +\frac{1}{2} k_n^T Q_{uu,n} k_n
\end{aligned}
$$

基準点からの変化量を確認するため、$\delta X_n=0, \delta u_n = 0$とした$\tilde{Q}$は

$$
\begin{aligned}
\tilde{Q}_n (\delta X_n = 0, \delta u_n = 0)&= \bar{Q}_n 
\end{aligned}
$$

よって、$\delta u_n = k_n$による$\tilde{Q}_n$の変化量は以下となる。

$$
\Delta \tilde{Q}_n = Q_{u,n}^T k_n +\frac{1}{2} k_n^T Q_{uu,n} k_n
$$

ここで$k_n=-Q_{uu,n}^{-1} Q_{u,n}$である。

- 一次項

$k_n$を適用すると以下となる。

$$
Q_{u,n}^T k_n = - c_n , \quad c_n = Q_{u,n}^T Q_{uu,n}^{-1} Q_{u,n}
$$

Step5 では$Q_{uu,n}$が正定値行列($Q_{uu,n} \succ 0 $)であると仮定したので、このStep6でも$Q_{uu,n}$は正定値行列であると仮定する。すると、その逆行列も正定値行列である。

$$
Q_{uu,n}^{-1} \succ 0
$$

よって、$Q_{u,n} \neq 0 $であれば、 $Q_{u,n}^T Q_{uu,n}^{-1} Q_{u,n} = c_n > 0$ である。

よって、一次項は以下となる。

$$
Q_{u,n}^T k_n = - c_n
$$

- 二次項

$$
u_n^{\text{new}} = \bar{u}_n + \alpha k_n + K_n \delta X_n
$$

$k_n$を適用すると以下となる。

$$
\begin{aligned}
\frac{1}{2} k_n^T Q_{uu,n} k_n &= \frac{1}{2}(-Q_{uu,n}^{-1} Q_{u,n})^T Q_{uu,n} (-Q_{uu,n}^{-1} Q_{u,n}) \\
&= \frac{1}{2} Q_{u,n}^T Q_{uu,n}^{-1} Q_{uu,n} (Q_{uu,n}^{-1} Q_{u,n}) \\
&= \frac{1}{2} Q_{u,n}^T Q_{uu,n}^{-1} Q_{u,n} \\
&= \frac{1}{2} c_n
\end{aligned}
$$

よって、$\Delta \tilde{Q}_n$は$k_n$の入力により負となるため、$k_n$は$\tilde{Q}_n$を低下させる方向に入力を与える。

$$
\Delta \tilde{Q}_n = -c_n +\frac{1}{2} c_n = -\frac{1}{2} c_n < 0
$$

$K_n$ は状態摂動 $\delta X_n$ が存在する場合に入力を修正する。

またStep 5 で求めたように、$Q_{uu,n} \succ 0$である場合、$\delta u_n$は局所二次Q関数を最小化する一意な入力摂動である。

まとめると、

- $Q_{uu,n} \succ 0$のとき、$k_n$は状態摂動$\delta X_n = 0$ とした局所二次Q関数$\tilde{Q}_n$を一意に最小化する入力摂動である。
- $Q_{u,n} \neq 0$ なら、$k_n$を適用した局所二次Q関数の変化量は以下となる。
    $$
    \Delta \tilde{Q}_n = -\frac{1}{2} c_n < 0
    $$
    したがって、$k_n$は基準点から局所二次Q関数を低下させる方向へ入力を修正する。


#### $k_n$の調整について

入力修正則 $\delta u_n^\star $は局所二次Q関数を最小化する修正量だが、この入力修正量により計算される候補軌道で総コストは局所二次関数ではなく、元の非線形な関数を用いて計算が行われる。この局所二次関数と元の非線形関数との差により、総コストが必ず減少するとは限らない。<br>
また、大きな$\delta u_n\star$で入力が修正される場合、基準点から大きく外れた候補点となる可能性がある。この場合、局所二次Q関数は元の非線形Q関数を近似することが出来なくなり、もともと扱いたかった非線形問題を扱うことが出来ず、異なる問題を解いていることにもなる。

そこで、局所二次近似が元の非線形Q関数のコスト変化を十分に予測できる範囲で、元の非線形問題の総コストが実際に減少する候補軌道を探すため、$\delta u_n^\star$の係数の$k_n$にLine Search係数$\alpha$を掛け、候補入力を次のように構成する。

$$
u_n^{\text{new}} = \bar{u}_n + \alpha k_n + K_n \delta X_n
$$


#### $K_n$に係数を掛けない理由

forward pass で入力摂動を以下のように設定して、計算を進める。解析のため、$\alpha$を十分に小さくと考え、局所近似式が元の非線形の式を十分に表すことが出来ていると考える。

$$
\delta u_n = \alpha k_n + K_n \delta X_n
$$

すると、forward passでは次のように状態を求めることが出来る。

$$
\begin{aligned}
\delta X_{n+1} &= A_n \delta X_n + B_n \delta u_n \\
&= A_n \delta X_n + B_n (\alpha k_n + K_n \delta X_n) \\
&= (A_n + B_n K_n) \delta X_n + B_n \alpha k_n
\end{aligned}
$$

- 時刻 $0$

初期状態は基準状態と候補状態が一致しているので $\delta X_0 = 0$ となる。

よって、入力修正項による状態摂動は次のようになる。

$$
\delta X_1 = \alpha B_0 k_0
$$

- 時刻 $1$

時刻$0$で求めた状態摂動$\delta X_1$ より、入力修正項 $\delta u_1$ は次のようになる。

$$
\delta u_1 = \alpha k_1 + \alpha (B_0 k_0) K_1
$$

よって、$\delta X_1$を介して$K_1$も$k_1$と同様に$\alpha$で調整が行われているといえる。これがforward pass で計算する候補軌道全体に伝わっていく。


もし、調整項として$\beta$を$K_n$に掛けているなら、入力修正項 $\delta \tilde{u}_1$は次のようになり、過剰な調整が行われることになる。

$$
\delta \tilde{u}_1 = \alpha k_1 + \alpha \beta (B_0 k_0) K_1
$$

これが、$K_n$にはLine Search係数$\alpha$をかけない理由である。

もちろん非線形なモデルでは上記のように簡単には解析できないが、状態摂動を介して$\alpha$が$K_n$に伝わることは同様に行われている。

#### $\alpha$の調整範囲

この$\alpha$を付けた$\delta u_n = \alpha k_n$による局所二次Q関数の変化量$\Delta \tilde{Q}_n$ は次のようになる。

$$
\Delta \tilde{Q}_n = -\alpha c_n +\frac{1}{2} \alpha^2 c_n = -\alpha(1 - \frac{\alpha}{2}) c_n < 0 , \quad 0 < \alpha < 2
$$

通常 $\alpha$ の範囲は [0, 1] であるため、この範囲であれば入力摂動により$\tilde{Q}_n$を減少させることが出来る。

そして$\alpha$での両端では次のような意味となる。

- $\alpha = 1$ : $\tilde{Q}_n$の減少幅が大きいが、局所二次近似$\tilde{Q}_n$は元の非線形Q関数を表す精度が荒くなる可能性がある
- $0 < \alpha \ll 1$ : $\tilde{Q}_n$の減少幅は小さいが、局所二次近似$\tilde{Q}_n$は元の非線形Q関数をよく表している。


$\alpha$をこの間で調整するため、以下のように1から半分ずつ調整していき、ゼロにならない範囲で、それぞれでforward passによる候補軌道を計算する。

$$
\alpha \in \{1.0, 0.5, 0.25, 0.125,\cdots \}
$$



## 3. 全体の流れ

forward pass と Line Searchの流れを説明する。

- 前提

前回の反復で基準軌道 ($\bar{X}^{(i)}, \bar{U}_^{(i)}$) が求められているとする。

そして、基準軌道による総コストを$\bar{J}$とする。

Line Search係数$alpha=1$とする。

- 1. forward pass と総コスト計算

入力修正則の$alpha$を設定し、forward pass で候補軌道 ((${X}^{\text{new}}, \bar{U}_^{\text{new}}$)) を求める。

そして、その候補軌道による総コスト $J^{\text{new}}$ を計算する。

- 2. $\bar{J}$と$J^{\text{new}}$ の比較

$\bar{J} < J^{\text{new}}$ である場合、その候補軌道を次の反復の基準軌道として設定する。

$\bar{J} \ge J^{\text{new}}$ である場合、$alpha = 0.5 \times \alpha$として、1へ戻り、forward pass を計算する。

繰り返しの数がある一定を超えた場合、

chatgpt どうなるんだ?

## 4. forward pass の計算手順

### 時刻 $n=0$

初期状態を以下のように設定する。

$$
X_0^{\text{new}} = \bar{X}_0
$$

候補状態と基準状態の差は次のようにゼロとなる。

$$
\delta X_0^{\text{new}} = X_0^{\text{new}} - \bar{X}_0 = 0
$$

新しい入力は次のように計算される。

$$
\begin{aligned}
u_0^{\text{new}} &= \bar{u}_0 + \alpha k_0 + K_0 \delta X_0^{\text{new}} \\
&= \bar{u}_0 + \alpha k_0
\end{aligned}
$$

これを非線形動力学モデルへ代入し、次の時刻の状態を算出する。

$$
X_1^{\text{new}} = f(X_0^{\text{new}}, u_0^{\text{new}})
$$

### 時刻 $n=1$

候補状態と状態状態の差は次のようになる。

$$
\delta X_1^{\text{new}} = X_1^{\text{new}} - \bar{X}_1
$$

新しい入力は次のように計算される。

$$
u_1^{\text{new}} = \bar{u}_1 + \alpha k_1 + K_1 \delta X_1^{\text{new}} \\
$$

これを非線形動力学モデルへ代入し、次の時刻の状態を算出する。

$$
X_2^{\text{new}} = f(X_1^{\text{new}}, u_1^{\text{new}})
$$

### 時刻 $n$

候補状態と状態状態の差は次のようになる。

$$
\delta X_n^{\text{new}} = X_n^{\text{new}} - \bar{X}_n
$$

新しい入力は次のように計算される。

$$
u_n^{\text{new}} = \bar{u}_n + \alpha k_n + K_n \delta X_n^{\text{new}} \\
$$

これを非線形動力学モデルへ代入し、次の時刻の状態を算出する。

$$
X_{n+1}^{\text{new}} = f(X_n^{\text{new}}, u_n^{\text{new}})
$$
